## Section 1: Load Final Artifacts

In [1]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb

In [3]:
ranker = joblib.load(
    "../artifacts/final_ranker.pkl"
)

FEATURE_COLUMNS = joblib.load(
    "../artifacts/final_ranker_features.pkl"
)

In [4]:
interaction_df = pd.read_parquet(
    "../data/processed/model_df.parquet"
)

product_features = pd.read_parquet(
    "../artifacts/product_features.parquet"
)

popularity_df = pd.read_parquet(
    "../artifacts/popularity_recommender.parquet"
)

## Section 2: Rebuild Customer Features

In [5]:
customer_spend = (
    interaction_df
    .groupby("customer_unique_id")["price"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "price": "customer_total_spend"
        }
    )
)

In [6]:
customer_frequency = (
    interaction_df
    .groupby("customer_unique_id")
    .size()
    .reset_index(name="customer_purchase_count")
)

In [7]:
customer_category = (
    interaction_df
    .groupby(
        [
            "customer_unique_id",
            "product_category_name"
        ]
    )
    .size()
    .reset_index(name="cnt")
)

In [8]:
customer_favorite_category = (
    customer_category
    .sort_values(
        "cnt",
        ascending=False
    )
    .drop_duplicates(
        subset=["customer_unique_id"]
    )
)

In [9]:
customer_favorite_category = (
    customer_favorite_category[
        [
            "customer_unique_id",
            "product_category_name"
        ]
    ]
    .rename(
        columns={
            "product_category_name":
            "favorite_category"
        }
    )
)

In [10]:
customer_features_eval = (
    customer_spend
    .merge(
        customer_frequency,
        on="customer_unique_id"
    )
    .merge(
        customer_favorite_category,
        on="customer_unique_id"
    )
)

## Section 3: Build Product Table

In [11]:
product_features_eval = (
    product_features
    .merge(
        popularity_df[
            [
                "product_id",
                "popularity_score"
            ]
        ],
        on="product_id",
        how="left"
    )
)

In [12]:
product_features_eval[
    "popularity_score"
] = (
    product_features_eval[
        "popularity_score"
    ].fillna(0)
)

## Section 4: Recommendation Function

In [13]:
def recommend_for_customer(
    customer_id,
    top_k=10
):

    customer_row = customer_features_eval[
        customer_features_eval[
            "customer_unique_id"
        ] == customer_id
    ]

    if customer_row.empty:
        return None

    customer_row = customer_row.iloc[0]

    purchased_products = set(
        interaction_df[
            interaction_df[
                "customer_unique_id"
            ] == customer_id
        ]["product_id"]
    )

    candidates = product_features_eval[
        ~product_features_eval[
            "product_id"
        ].isin(
            purchased_products
        )
    ].copy()

    candidates[
        "customer_total_spend"
    ] = customer_row[
        "customer_total_spend"
    ]

    candidates[
        "customer_purchase_count"
    ] = customer_row[
        "customer_purchase_count"
    ]

    candidates[
        "is_favorite_category_match"
    ] = (
        candidates[
            "product_category_name"
        ]
        ==
        customer_row[
            "favorite_category"
        ]
    ).astype(int)

    X = candidates[
        FEATURE_COLUMNS
    ]

    candidates["score"] = ranker.predict(X)

    return (
        candidates
        .sort_values(
            "score",
            ascending=False
        )
        .head(top_k)
    )

## Section 5: Pick Evaluation Customers

In [14]:
top_customer = (
    customer_spend
    .sort_values(
        "customer_total_spend",
        ascending=False
    )
    .iloc[0]["customer_unique_id"]
)

top_customer

'0a0a92112bd4c708ca5fde585afaa872'

In [15]:
medium_customer = (
    customer_spend[
        customer_spend[
            "customer_total_spend"
        ].between(
            100,
            200
        )
    ]
    .sample(1, random_state=42)
    .iloc[0]["customer_unique_id"]
)

medium_customer

'8ce938b7447a8327dadf2a50c43ac271'

In [16]:
low_customer = (
    customer_spend[
        customer_spend[
            "customer_total_spend"
        ] < 50
    ]
    .sample(1, random_state=42)
    .iloc[0]["customer_unique_id"]
)

low_customer

'daaa4bd4b2ac37cd00efc852878b626a'

## Section 6: Inspect Their Purchase History

In [17]:
interaction_df[
    interaction_df[
        "customer_unique_id"
    ] == top_customer
][
    [
        "product_id",
        "product_category_name",
        "price"
    ]
]

,product_id,product_category_name,price
4243,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4244,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4245,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4246,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4247,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4248,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4249,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0
4250,5769ef0a239114ac3a854af00df129e4,telefonia_fixa,1680.0


## Section 7: Generate Recommendations

In [19]:
recommend_for_customer(
    top_customer
)[
    [
        "product_category_name",
        "popularity_score",
        "score"
    ]
]

,product_category_name,popularity_score,score
8138,moveis_escritorio,0.022989,1.384861
18016,esporte_lazer,0.011494,0.974921
5675,beleza_saude,0.022989,0.701536
24029,beleza_saude,0.013410,0.473677
26862,industria_comercio_e_negocios,0.011494,0.326076
32262,casa_construcao,0.015326,0.294645
8290,informatica_acessorios,0.620690,-0.088967
15623,telefonia_fixa,0.019157,-0.576749
32098,ferramentas_jardim,0.683908,-0.585106
793,relogios_presentes,0.595785,-0.665430


In [20]:
recommend_for_customer(
    medium_customer
)[
    [
        "product_category_name",
        "popularity_score",
        "score"
    ]
]

,product_category_name,popularity_score,score
4855,informatica_acessorios,0.057471,1.759134
29129,informatica_acessorios,0.490421,1.646818
21395,informatica_acessorios,0.333333,1.548700
2795,informatica_acessorios,0.233716,1.478536
26252,informatica_acessorios,0.172414,1.249059
8290,informatica_acessorios,0.620690,1.095987
10963,informatica_acessorios,0.197318,0.845286
7818,informatica_acessorios,0.034483,0.761439
15932,informatica_acessorios,0.153257,0.694853
6869,informatica_acessorios,0.237548,0.600428


In [21]:
recommend_for_customer(
    low_customer
)[
    [
        "product_category_name",
        "popularity_score",
        "score"
    ]
]

,product_category_name,popularity_score,score
17089,informatica_acessorios,0.030651,1.367906
27184,informatica_acessorios,0.013410,1.155295
28309,informatica_acessorios,0.057471,0.920504
13161,informatica_acessorios,0.009579,0.907979
12853,informatica_acessorios,0.009579,0.907979
12522,informatica_acessorios,0.045977,0.888524
14227,informatica_acessorios,0.030651,0.835936
18592,informatica_acessorios,0.040230,0.784761
15226,informatica_acessorios,0.038314,0.729562
10136,informatica_acessorios,0.032567,0.724620


In [22]:
print("TOP CUSTOMER")
display(
    interaction_df[
        interaction_df["customer_unique_id"] == top_customer
    ][
        ["product_category_name", "price"]
    ]
    .sort_values("price", ascending=False)
)

TOP CUSTOMER


,product_category_name,price
4243,telefonia_fixa,1680.0
4244,telefonia_fixa,1680.0
4245,telefonia_fixa,1680.0
4246,telefonia_fixa,1680.0
4247,telefonia_fixa,1680.0
4248,telefonia_fixa,1680.0
4249,telefonia_fixa,1680.0
4250,telefonia_fixa,1680.0


In [23]:
print("MEDIUM CUSTOMER")
display(
    interaction_df[
        interaction_df["customer_unique_id"] == medium_customer
    ][
        ["product_category_name", "price"]
    ]
    .sort_values("price", ascending=False)
)

MEDIUM CUSTOMER


,product_category_name,price
31604,informatica_acessorios,105.0


In [24]:
print("LOW CUSTOMER")
display(
    interaction_df[
        interaction_df["customer_unique_id"] == low_customer
    ][
        ["product_category_name", "price"]
    ]
    .sort_values("price", ascending=False)
)

LOW CUSTOMER


,product_category_name,price
3536,informatica_acessorios,43.89
